In [6]:
using Random
using LinearAlgebra
# Fidelity of Spin operators
# For n spin density matrices we have 
# we represent a density matrix by 4^n-1 real or complex numbers (should be real, but who knows)
# the indexes represent the spin configuration, 
# i.e. the first few for n=4: [1,1,1,x], [1,1,1,y], [1,1,1,z], [1,1,x,1], [1,1,x,x], [1,1,x,y]... (where one represents an identity matrix)
# Generate a density matrix (random, trace 1, hermitian, positive semidefinite)
function Random_Density_Matrix(n::Int; rng = Random.MersenneTwister(1234))
    # generate random matrix A 
    d = 2^n
    A::Matrix{ComplexF64} = rand(rng, d,d) + im*rand(rng, d,d)
    # make it hermitian
    B = (A + A')
    # make it positive semidefinite
    C = B'*B
    # normalize it
    D = C/LinearAlgebra.tr(C)
    return D
end
# Generate set of all Pauli matrices of n qubits
# Generate expectation values of Pauli Operators with respect to a density matrix
function Expectation_Values(pauli_operators, density_matrix::Matrix{ComplexF64})
    d = size(density_matrix)[1]
    pref = 1.0/d
    expectation_values::Vector{ComplexF64} = []
    #expectation_values::Vector{Float64} = []
    for pauli_operator in pauli_operators[2:end]
        push!(expectation_values, pref*real(LinearAlgebra.tr(pauli_operator*density_matrix)))
        #push!(expectation_values, LinearAlgebra.tr(pauli_operator*density_matrix))
    end
    return expectation_values
end

using SparseArrays
# generate a set of n qubit pauli operators in sparse form
function All_sparse_Pauli_Operators(n::Int)::Vector{SparseMatrixCSC}
    I = sparse([1 0; 0 1])
    X = sparse([0 1; 1 0])
    Y = sparse([0 -im; im 0])
    Z = sparse([1 0; 0 -1])
    pauli_operators = [I, X, Y, Z]
    if n == 1
        return pauli_operators
    else
        lower_pauli_operators = All_sparse_Pauli_Operators(n-1)
        new_pauli_operators::Vector{SparseMatrixCSC} = []
        for pauli_operator in pauli_operators
            for lower_pauli_operator in lower_pauli_operators
                push!(new_pauli_operators, kron(pauli_operator, lower_pauli_operator))
            end
        end
        return new_pauli_operators
    end
end
# generate Density matrix from expectation values
function Sparse_Matrix_from_expectation_Values(expectation_values, pauli_operators::Vector{SparseMatrixCSC})
    # generate matrix from expectation values
    n = Int(log(4,length(expectation_values)+1))
    d = 2^n
    weight = 1.0/d
    matrix= diagm(fill(weight, d))
    for (i, pauli_operator) in enumerate(pauli_operators[2:end])
        matrix += expectation_values[i]*pauli_operator
    end
    return matrix
end
function Fidelity_from_expectation_values(sigma, sqrt_rho::Matrix{ComplexF64}, pauli_operators::Vector{SparseMatrixCSC}) # sqrt_rho is the target
    # Calculate abs(Tr(sqrt(sqrt(rho)*sigma*sqrt(rho))))^2
    sigma_mat = Sparse_Matrix_from_expectation_Values(sigma, pauli_operators)
    F = abs(LinearAlgebra.tr(sqrt(sqrt_rho*sigma_mat*sqrt_rho)))^2
    if F > 1.0 # get rid of a little bit of noise, 
        F = 1.0
        #F = 2 - F   #remove the noise from 1.0 ? 
    end
    return F
end
function Infidelity_from_expectation_values(sigma, sqrt_rho::Matrix{ComplexF64}, pauli_operators::Vector{SparseMatrixCSC}) # sqrt_rho is the target
    return 1-Fidelity_from_expectation_values(sigma, sqrt_rho, pauli_operators)
end

# Test
m = 2
density_matrix = Random_Density_Matrix(m)
pauli_operators = All_sparse_Pauli_Operators(m)
expectation_values = Expectation_Values(pauli_operators, density_matrix)
# check if rho_matrix is equal to density_matrix
Fidelity_from_expectation_values(expectation_values, sqrt(density_matrix), pauli_operators)

1.0

In [7]:
using LinearAlgebra

function sqrt_positive_semidefinite_matrix(hermitian_matrix)#::Matrix{ComplexF64})
    # Perform eigen decomposition
    eigvals, eigvecs = eigen(hermitian_matrix)
    eigvals = real(eigvals)
    # Compute the square roots of the eigenvalues, treating negative values as complex
    sqrt_eigvals = sqrt.(real.(eigvals))

    # Reconstruct the square root of the matrix
    sqrt_matrix = eigvecs * Diagonal(sqrt_eigvals) * eigvecs'

    return sqrt_matrix
end

function Sparse_Matrix_from_expectation_Values(expectation_values, pauli_operators::Vector{SparseMatrixCSC})
    # generate matrix from expectation values
    n = Int(log(4,length(expectation_values)+1))
    d = 2^n
    weight = 1.0/d
    matrix= diagm(fill(weight, d))
    for (i, pauli_operator) in enumerate(pauli_operators[2:end])
        matrix += expectation_values[i]*pauli_operator
    end
    return matrix
end
function Fidelity_from_expectation_values(sigma, sqrt_rho::Matrix{ComplexF64}, pauli_operators::Vector{SparseMatrixCSC}) # sqrt_rho is the target
    # Calculate abs(Tr(sqrt(sqrt(rho)*sigma*sqrt(rho))))^2
    sigma_mat = Sparse_Matrix_from_expectation_Values(sigma, pauli_operators)
    F = abs(LinearAlgebra.tr(sqrt_positive_semidefinite_matrix(sqrt_rho*sigma_mat*sqrt_rho)))^2
    if F > 1.0 # get rid of a little bit of noise, 
        F = 1.0
        #F = 2 - F   #remove the noise from 1.0 ? 
    end
    return F
end
function Infidelity_from_expectation_values(sigma, sqrt_rho::Matrix{ComplexF64}, pauli_operators::Vector{SparseMatrixCSC}) # sqrt_rho is the target
    return 1-Fidelity_from_expectation_values(sigma, sqrt_rho, pauli_operators)
end
# Example usage
m = 4
rng = Random.MersenneTwister(100)
rho = Random_Density_Matrix(m, rng=rng)
sigma = Random_Density_Matrix(m, rng=rng)
sqrt_sigma = sqrt_positive_semidefinite_matrix(sigma)
pauli_operators = All_sparse_Pauli_Operators(m)
expectation_values = Expectation_Values(pauli_operators, rho)
expectation_values = real(expectation_values)

Infidelity_from_expectation_values(expectation_values, sqrt_sigma, pauli_operators)

using Zygote
diff_cost_function = gradient(x -> Infidelity_from_expectation_values(x, sqrt_sigma, pauli_operators), expectation_values)


([0.3246555805376856, 4.431814366647562, 9.10368244323101, -10.274767217140607, -0.28768922246965967, 2.53600427095518, 7.579926241485284, -7.810056109438116, 1.1574868698841776, -5.062234727084874  …  2.421708064797734, 2.5646044388361533, 7.057377445962245, 4.42791580390683, 5.135595796619818, -0.005251100315031509, -9.800866044184156, 2.3771585849695986, 10.62548828633945, -0.8533193602548947],)

In [8]:
using BenchmarkTools
function create_diff_function(sqrt_sigma, pauli_operators)
    return x -> gradient(x -> Infidelity_from_expectation_values(x, sqrt_sigma, pauli_operators), x)[1]
end
grad_fun = create_diff_function(sqrt_sigma, pauli_operators)
@benchmark grad_fun(expectation_values)

BenchmarkTools.Trial: 30 samples with 1 evaluation.
 Range (min … max):  165.775 ms … 171.356 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     167.949 ms               ┊ GC (median):    1.39%
 Time  (mean ± σ):   167.990 ms ±   1.204 ms  ┊ GC (mean ± σ):  1.27% ± 0.39%

                 █▃     ▃▃▃  ▃     ▃                             
  ▇▁▁▁▇▁▁▁▇▇▁▁▁▇▁██▇▁▁▇▁███▇▇█▁▇▁▇▁█▁▁▁▁▁▁▁▇▁▁▁▇▁▁▁▇▁▁▁▁▁▁▁▁▁▁▇ ▁
  166 ms           Histogram: frequency by time          171 ms <

 Memory estimate: 41.13 MiB, allocs estimate: 275184.

In [ ]:
## Functions to construct expectation values and figures of merit 
# we assume n qubits and one cavity qubit (qubitization through number operator)
function Unitary_on_rho(U_qubit, rho_qubit, n_qubits, rho_cavity) 
    # unitary rotation only on each of the qubits, not on cavity, tensor product state at the beginning
    # first construc density matrix from rho_qubit and rho_cavity
    # rho_qubit times n_qubits times rho_cavity
    function n_kron(A, n)
        if n == 1
            return A
        else
            return kron(A, n_kron(A, n-1))
        end
    end
    rho = kron(n_kron(rho_qubit, n_qubits), rho_cavity)
    # generate unitary for all
    U = kron(n_kron(U_qubit, n_qubits), Diagonal([1.0, 1.0])) # 
    return unitary*rho*unitary'
end

In [1]:
# Operators
function a(d::Int)
    a = zeros(d,d)
    for i in 1:d-1
        a[i,i+1] = sqrt(i)
    end
    return a
end
function a_dag(d::Int)
    a_dag = zeros(d,d)
    for i in 2:d
        a_dag[i,i-1] = sqrt(i-1)
    end
    return a_dag
end
function n(d::Int)
    n = zeros(d,d)
    for i in 1:d
        n[i,i] = i-1
    end
    return n
end
# Test
# Generate density matrices for a bosonic mode
d = 4 # dimension of the Hilbert space
# show the operators
a(d)

4×4 Matrix{Float64}:
 0.0  1.0  0.0      0.0
 0.0  0.0  1.41421  0.0
 0.0  0.0  0.0      1.73205
 0.0  0.0  0.0      0.0

In [2]:
d = 7
A = zeros(d,d)
for y in 0:d-1
    for x in 0:d-1
        A[x+1, y+1] = sum([a_dag(d)^(m+y)*a(d)^(m+x)*(-1)^m/factorial(m)/sqrt(factorial(m+x))/sqrt(factorial(m+y)) for m in 0:d-1])[y+1,x+1]
    end
end
A

7×7 Matrix{Float64}:
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0
 1.0  1.0  1.0  1.0  1.0  1.0  1.0

In [18]:
d = 4
x = 1
y = 1
sum([a_dag(d)^(m+y)*a(d)^(m+x)*(-1)^m/factorial(m)/sqrt(factorial(m+x))/sqrt(factorial(m+y)) for m in 0:d-1])

4×4 Matrix{Float64}:
 0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0
 0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.5

In [19]:
a(2)

2×2 Matrix{Float64}:
 0.0  1.0
 0.0  0.0

In [24]:
## sum_k=0^d k!
#for d in 1:10
#    println(factorial(d))
#end

In [121]:

# Optimize expectation values to maximize fidelity
# test if we can optimize through the functions
# generate a random density matrix
m = 2
rho = Random_Density_Matrix(m)
sigma = Random_Density_Matrix(m)
sqrt_rho = sqrt(rho)
# generate pauli operators
pauli_operators = All_sparse_Pauli_Operators(m)
# generate initial expectation values
expectation_values = Expectation_Values(pauli_operators, sigma)
expectation_values = real(expectation_values)
# generate initial SpinDensityMatrix
using Optim
# use infidelity as cost function
intermediate_results = []
function cost_function(expectation_values)
    I =  Infidelity_from_expectation_values(expectation_values, sqrt_rho, pauli_operators)
    push!(intermediate_results, I)
    return I
end
# autodiff cost_function
using Zygote
diff_cost_function = gradient(x -> cost_function(x), expectation_values)

## optimize
#result = optimize(cost_function, expectation_values, LBFGS(), autodiff=:forward)
## check if result is correct
#using Plots
#plot(intermediate_results, label="infidelity")

LoadError: Mutating arrays is not supported -- called push!(Vector{Any}, ...)
This error occurs when you ask Zygote to differentiate operations that change
the elements of arrays in place (e.g. setting values with x .= ...)

Possible fixes:
- avoid mutating operations (preferred)
- or read the documentation and solutions for this error
  https://fluxml.ai/Zygote.jl/latest/limitations
